# Homework 3 Solutions

1. Download daily price data for QQQ, TLT, GLD, RWO from yahoo finance since 2016-01-01. Using the adjusted close price data, compute daily returns. This should be a DataFrame with index=date, columns=ticker and values=daily returns.

In [6]:
import yfinance as yf

In [7]:
univ = ['QQQ','TLT','GLD','RWO']
px = yf.download(univ, start="2016-01-01")

[*********************100%***********************]  4 of 4 completed


In [9]:
adj_close = px['Close']
ret = adj_close / adj_close.shift() - 1

2. Compute the signal. Complete the compute_momentum function which computes a simple momentum signal. The function takes in a DataFrame with index = date, columns=ticker and values containing daily returns. It returns a new DataFrame with index = date, columns=ticker and values containing the momentum signal for the ticker on that day. The momentum signal for each ticker is defined as the annualized sharpe ratio of the past 252 days.

In [10]:
import math

def compute_momentum(ret):
    momentum = ret.rolling(252).mean()/ret.rolling(252).std()*math.sqrt(252)
    return momentum

momentum = compute_momentum(ret)
momentum

Ticker,GLD,QQQ,RWO,TLT
Date,,,,
2016-01-04,NaN,NaN,NaN,NaN
2016-01-05,NaN,NaN,NaN,NaN
2016-01-06,NaN,NaN,NaN,NaN
2016-01-07,NaN,NaN,NaN,NaN
2016-01-08,NaN,NaN,NaN,NaN
...,...,...,...,...
2025-12-15,2.454387,0.734098,NaN,-0.020485
2025-12-16,2.540670,0.769812,NaN,0.124584
2025-12-17,2.653533,0.658066,NaN,0.201838


3. Create a portfolio. Complete the function compute_portfolio. This function takes as input the DataFrame "momentum" from above. It returns a new DataFrame "portfolio" which has the same index/columns and has as values portfolio weights. The weights are computed as follows. On each date, equal-weight long the tickers with a momentum signal above 1.

In [11]:
def compute_portfolio(momentum):
    portfolio = (momentum > 1)*1
    portfolio = portfolio.div(portfolio.abs().sum(1),0)
    return portfolio

portfolio = compute_portfolio(momentum)

In [14]:
compute_portfolio(momentum).dropna()

Ticker,GLD,QQQ,RWO,TLT
Date,,,,
2017-01-06,0.0,1.0,0.0,0.0
2017-01-09,0.0,1.0,0.0,0.0
2017-01-10,0.0,1.0,0.0,0.0
2017-01-11,0.0,1.0,0.0,0.0
2017-01-12,0.0,1.0,0.0,0.0
...,...,...,...,...
2025-12-15,1.0,0.0,0.0,0.0
2025-12-16,1.0,0.0,0.0,0.0
2025-12-17,1.0,0.0,0.0,0.0


4. Portfolio returns. Using the "portfolio" returned in part(3) and the returns generated in part(1), compute the returns to the simple momentum strategy. 
- What is the annualized sharpe ratio of the strategy? 
- How about the annualized sharpe ratio within each year? 
- How correlated is the strategy with the underlying tickers?
- Plot the cumulative sum of the returns through time

In [ ]:
strat_ret = (portfolio.shift()*ret).sum(1)
# minor, but should start returns when signal starts
strat_ret = strat_ret.loc[momentum.dropna().index[0]:] 

In [ ]:
# sharpe
strat_ret.mean()/strat_ret.std()*math.sqrt(252)

In [ ]:
# sharpe within each year
import numpy as np 

sharpe = lambda x: x.mean()/x.std()*math.sqrt(252) 
strat_ret.groupby([x.year for x in strat_ret.index]).apply(sharpe)

In [ ]:
# correlation
ret.corrwith(strat_ret)

In [ ]:
# cumulative returns
strat_ret.cumsum().plot()